﻿# 00 Descarga Universo - Checklist de Certificación (End-to-End)

Este documento está diseñado para ejecutarse como guía de notebook: cada bloque tiene:
- una **celda markdown** (qué valida y por qué)
- una **celda código** (cómo demostrarlo)

Objetivo: saber exactamente qué data se descargó, su cobertura, su integridad y si se puede continuar.

---


## 1) Verificar artefactos base del Universo PTI
Validamos existencia de outputs principales del universo PTI y metadatos de corrida.


In [1]:
from pathlib import Path

outdir = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti")
required = [
    "build_universe_pti.progress.json",
    "build_universe_pti.meta.json",
    "qa_coverage_by_cut.csv",
    "tickers_all.parquet",
    "tickers_active.parquet",
    "tickers_inactive.parquet",
]
for x in required:
    p = outdir / x
    print(x, "->", p.exists())


build_universe_pti.progress.json -> True
build_universe_pti.meta.json -> True
qa_coverage_by_cut.csv -> True
tickers_all.parquet -> True
tickers_active.parquet -> True
tickers_inactive.parquet -> True


## 2) Verificar cierre de corrida PTI
La corrida debe estar cerrada (`status=completed`) y coherente con meta/QA/panel.


In [2]:
import json
import pandas as pd
import pyarrow.dataset as ds
from pathlib import Path

outdir = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti")
progress = json.loads((outdir / "build_universe_pti.progress.json").read_text(encoding="utf-8"))
meta = json.loads((outdir / "build_universe_pti.meta.json").read_text(encoding="utf-8"))
qa = pd.read_csv(outdir / "qa_coverage_by_cut.csv")

panel = ds.dataset(str(outdir / "tickers_panel_pti"), format="parquet").to_table(columns=["snapshot_date","entity_id"]).to_pandas()
panel["snapshot_date"] = pd.to_datetime(panel["snapshot_date"], errors="coerce")

print("progress.status:", progress.get("status"))
print("progress snapshot:", progress.get("snapshot_date"), progress.get("snapshot_index"), "/", progress.get("snapshot_total"))
print("meta successful/requested:", meta.get("successful_cuts"), "/", meta.get("requested_cuts"))
print("panel rows:", len(panel), "snapshots:", panel["snapshot_date"].nunique(), "max:", panel["snapshot_date"].max())
print("qa rows:", len(qa), "qa max:", pd.to_datetime(qa["snapshot_date"], errors="coerce").max())


progress.status: completed
progress snapshot: 2026-03-09 7738 / 7738
meta successful/requested: 7738 / 7738
panel rows: 29735570 snapshots: 7738 max: 2026-03-09 00:00:00
qa rows: 7738 qa max: 2026-03-09 00:00:00


## 3) Auditor formal del universo PTI (go/no-go)
Ejecuta el agente auditor para certificar coherencia final de la corrida.


In [3]:
# Ejecutar en terminal:
# python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agent_universe_audit.py --outdir C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti --stale-minutes 10 --validate-latest-checkpoints 50 --out-report-json C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\00_data_certification\universe_audit_final.json


## 4) Construir y validar universo filtrado listed/deslisted (2005-2026)
Debe usar intersección temporal `[first_seen_date,last_seen_date]` con ventana objetivo.


In [4]:
# Ejecutar en terminal:
# python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\build_tickers_2005_2026.py

import pandas as pd
p = r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026.parquet"
d = pd.read_parquet(p)
print("rows:", len(d))
print(d[["ticker","first_seen_date","last_seen_date","status","primary_exchange"]].head(10).to_string(index=False))


rows:

 15979
ticker first_seen_date last_seen_date   status primary_exchange
   AAA      2005-01-01     2007-05-21 inactive             XNYS
  AACB      2025-04-07     2026-03-09   active             XNAS
  AACI      2024-08-15     2025-10-29 inactive             XNAS
  AACQ      2020-09-04     2021-06-24 inactive             XNAS
  AACT      2023-06-12     2025-09-24 inactive             XNYS
   AAC      2008-10-03     2010-02-07 inactive             XASE
   AAC      2021-03-25     2023-11-06 inactive             XNYS
   AAI      2005-01-01     2011-05-02 inactive             XNYS
   AAM      2024-09-16     2026-02-02 inactive             XNYS
   AAN      2009-04-21     2010-12-13 inactive             XNYS


## 5) Ejecutar enriquecimiento híbrido (as-of + catálogo inactivos)
Descarga atributos de referencia/fundamentales por ticker y aplica merge por prioridad de fuentes.


In [5]:
# Ejecutar en terminal:
# python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\build_universe_hybrid_enriched.py --input C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026.parquet --outdir C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\hybrid_enriched --batch-size 200 --inactive-catalog-refresh --resume


## 6) Validar cobertura de campos enriquecidos
Debemos medir cobertura por `final_*` y distribución por fuente (`source_final_*`).


In [6]:
import pandas as pd
from pathlib import Path

p = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\hybrid_enriched\universe_hybrid_enriched.parquet")
d = pd.read_parquet(p)

final_cols = [c for c in d.columns if c.startswith("final_")]
cov = pd.DataFrame({
    "field": final_cols,
    "non_null": [int(d[c].notna().sum()) for c in final_cols]
})
cov["rows"] = len(d)
cov["coverage_pct"] = (cov["non_null"] / cov["rows"] * 100).round(2)

display(cov.sort_values("coverage_pct", ascending=False).reset_index(drop=True))
print("source_final_delisted_utc:")
print(d["source_final_delisted_utc"].value_counts(dropna=False).to_string())


,field,non_null,rows,coverage_pct
0,final_type,15979,15979,100.00
1,final_primary_exchange,15979,15979,100.00
2,final_last_updated_utc,15979,15979,100.00
3,final_ticker_root,15978,15979,99.99
4,final_round_lot,15978,15979,99.99
5,final_cik,15385,15979,96.28
6,final_share_class_shares_outstanding,15245,15979,95.41
7,final_phone_number,11643,15979,72.86
8,final_address_city,11641,15979,72.85
9,final_address_address1,11641,15979,72.85


source_final_delisted_utc:
source_final_delisted_utc
inactive_catalog      7366
missing               5255
inferred_last_seen    3358


## 7) Validar semántica de deslistado (oficial vs inferido)
No tratar como equivalente `inactive_catalog` (oficial) y `inferred_last_seen` (inferencia).


In [7]:
import pandas as pd

p = r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\hybrid_enriched\universe_hybrid_enriched.parquet"
d = pd.read_parquet(p)

m = d[d["source_final_delisted_utc"].eq("missing")]
print("missing delisted rows:", len(m))
print("missing by status:")
print(m["status"].value_counts(dropna=False).to_string())
print("missing by exchange:")
print(m["primary_exchange"].value_counts(dropna=False).to_string())


missing delisted rows: 5255
missing by status:
status
active    5255
missing by exchange:
primary_exchange
XNAS    3281
XNYS    1739
XASE     235


## 8) Preparar universo canónico para fundamentals (UPPER + dedup)
Evita colisiones por case en filesystem Windows (`ADSw` vs `ADSW`).


In [8]:
import pandas as pd

src = r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026.parquet"
out = r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet"

u = pd.read_parquet(src)
u["ticker"] = u["ticker"].astype("string").str.strip().str.upper()
u = u.dropna(subset=["ticker"]).drop_duplicates(subset=["ticker"]).sort_values("ticker")
u.to_parquet(out, index=False)
print("rows:", len(u))
print("saved:", out)


rows: 12468
saved: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet


## 9) Descargar fundamentals por endpoint (en D:\financial)
Descarga por ticker para 4 datasets: income, balance, cashflow, ratios.


In [9]:
# Ejecutar en terminal:
# python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\download_fundamentals_v1.py --input C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet --outdir D:\financial --datasets income_statements,balance_sheets,cash_flow_statements,ratios --batch-size 200 --workers 12 --limit 1000 --max-pages 50 --resume --resume-validate --progress-every 25


## 10) Monitor de corrida fundamentals
Verificar progreso y errores/warnings.


In [10]:
# En terminal (loop monitor):
# while ($true) { if (Test-Path D:\financial\_run\download_fundamentals_v1.progress.json) { Get-Content D:\financial\_run\download_fundamentals_v1.progress.json -Raw } else { "waiting progress file..." }; Start-Sleep -Seconds 30 }


## 11) Auditar fundamentals de extremo a extremo
Cobertura por endpoint, integridad file-by-file, y rango temporal por ticker+endpoint.


In [11]:
# Ejecutar en terminal:
# python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_fundamentals\cell_code\audit_fundamentals_download.py --input-universe C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet --outdir D:\financial --datasets income_statements,balance_sheets,cash_flow_statements,ratios --limit 1000 --max-pages 50


## 12) Sonda de filtro server-side (5 tickers)
Prueba diagnóstica: detecta si el endpoint devuelve masivamente tickers distintos al solicitado.


In [12]:
# Ejecutar en terminal:
# python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_fundamentals\cell_code\probe_fundamentals_ticker_filter.py --tickers AAPL,MSFT,NVDA,JNJ,WMT --datasets income_statements,balance_sheets,cash_flow_statements,ratios --outdir D:\financial\_audit --limit 100 --max-pages 3


## 13) Evidencia final mínima para certificar (GO)
Para continuar, deben cumplirse simultáneamente:
1. Universo PTI: auditor final `overall=OK`.
2. Fundamentals: cobertura por endpoint = 100% sobre `tickers_2005_2026_upper.parquet`.
3. `file_level_audit`: sin issues severas (`read_error`, `ticker_col_mismatch`, `multi_cik`, `rows_hit_page_cap`, `tickers_field_mismatch`).
4. `date_ranges_by_ticker_endpoint.csv` completo (con `date_start/date_end` cuando hay datos).
5. Separación explícita entre dato oficial e inferido (`source_final_delisted_utc`).


In [13]:
import json
import pandas as pd
from pathlib import Path

audit = Path(r"D:\financial\_audit")
summary_p = audit / "audit_summary.json"
cov_p = audit / "coverage_by_endpoint.csv"
sev_p = audit / "severe_issues.csv"

if not summary_p.exists() or not cov_p.exists():
    print("AUDIT pendiente: no existen a?n audit_summary.json y/o coverage_by_endpoint.csv en", audit)
    print("Ejecuta primero:")
    print(r"python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_fundamentals\cell_code\audit_fundamentals_download.py --input-universe C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet --outdir D:\financial --datasets income_statements,balance_sheets,cash_flow_statements,ratios --limit 1000 --max-pages 50")
else:
    summary = json.loads(summary_p.read_text(encoding="utf-8"))
    cov = pd.read_csv(cov_p)
    sev = pd.read_csv(sev_p) if sev_p.exists() else pd.DataFrame()

    print("status:", summary.get("status"))
    print("coverage:")
    print(cov.to_string(index=False))
    print("severe_issues:", len(sev))




AUDIT pendiente: no existen a?n audit_summary.json y/o coverage_by_endpoint.csv en D:\financial\_audit
Ejecuta primero:
python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_fundamentals\cell_code\audit_fundamentals_download.py --input-universe C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet --outdir D:\financial --datasets income_statements,balance_sheets,cash_flow_statements,ratios --limit 1000 --max-pages 50


## 14) Próximo paso (Small-Cap PTI)
Solo si el punto 13 está en GO:
- construir `population_target_pti(date, figi, ticker, exchange, status, close_t, shares_outstanding_t, market_cap_t, is_small_cap_t)`
- con LOCF <= 180 días y sin look-ahead.


In [14]:
# Placeholder de launcher futuro:
# python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\build_population_target_pti.py --universe C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet --fundamentals-root D:\financial --outdir C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\population_target_pti --locf-days 180 --resume


## Opini?n Final (Dictamen)
**Estado actual:** `NO-GO` para pasar a fase siguiente sin cerrar auditor?a final de fundamentals.

**Qu? s? est? bien:**
- Universo PTI y flujo de enriquecimiento h?brido est?n estructurados y trazables.
- Se corrigieron problemas clave del downloader (normalizaci?n ticker, resume-validate, concurrencia, trazabilidad de filtros).

**Qu? falta para certificar al 100%:**
1. Ejecutar auditor?a completa de fundamentals y generar `D:\financial\_audit\audit_summary.json`.
2. Validar cobertura 100% por endpoint contra `tickers_2005_2026_upper.parquet`.
3. Validar `severe_issues.csv == 0` (sin mezcla de ticker, sin multi_cik, sin archivos sospechosos).
4. Mantener separaci?n expl?cita entre `delisted_utc` oficial vs inferido (`source_final_delisted_utc`).

**Criterio de continuaci?n (GO):**
- `audit_summary.status == "OK"`
- cobertura por endpoint = 100%
- issues severas = 0

Si esos 3 checks se cumplen, mi recomendaci?n es **GO** para construir `population_target_pti` (small-cap PTI sin look-ahead).
